# ImageJudge — Standalone Training, Optimization & Eval

Self-contained notebook for the Pixtral-based ImageJudge module.

**Evaluates:** whether a generated image is `aligned` or `not_aligned` with its `image_content` description  
**Input:** `image_content: str` + `image: dspy.Image` (PNG as base64)  
**Output:** `verdict` (`aligned` / `not_aligned`) + `reasoning`  
**Dataset:** `data/image_mcq/training_dataset_24_judge_ready.json` (24 questions x 2 candidates x 2 models = 96 examples)  
**Flow:** Setup -> Generate Images -> Define Models -> Baseline Eval -> GEPA Optimize -> Post-Eval -> Promptfoo

Run all cells top-to-bottom.

## Cell 1 — Setup & Imports

In [ ]:
import sys, os, json, subprocess, warnings, base64, io, urllib.request
from pathlib import Path
from collections import Counter

warnings.filterwarnings('ignore')

_candidate = Path().resolve()
if   (_candidate        / 'utils.py').exists(): PROJECT_ROOT = _candidate
elif (_candidate.parent / 'utils.py').exists(): PROJECT_ROOT = _candidate.parent
else: raise RuntimeError('Cannot locate project root. Open Jupyter from d:/Topin or d:/Topin/notebooks.')

NOTEBOOK_DIR  = PROJECT_ROOT / 'notebooks'
DATA_DIR      = PROJECT_ROOT / 'data' / 'image_mcq'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
EVALS_DIR     = NOTEBOOK_DIR / 'evals'
IMAGES_DIR    = DATA_DIR / 'images' / 'judge_training'

ARTIFACTS_DIR.mkdir(exist_ok=True)
EVALS_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

DATASET_PATH   = DATA_DIR / 'training_dataset_24_judge_ready.json'
OPTIMIZED_PATH = ARTIFACTS_DIR / 'image_judge_optimized.json'
EVAL_OUTPUT    = EVALS_DIR / 'image_judge_eval_output.json'

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

_venv_sp = PROJECT_ROOT / '.venv' / 'Lib' / 'site-packages'
if not _venv_sp.exists():
    _venv_sp = PROJECT_ROOT / '.venv' / 'lib' / 'site-packages'
if _venv_sp.exists() and str(_venv_sp) not in sys.path:
    sys.path.insert(0, str(_venv_sp))
    print(f'Injected venv site-packages: {_venv_sp}')

import dspy

print(f'Project root  : {PROJECT_ROOT}')
print(f'Dataset       : {DATASET_PATH}')
print(f'Images dir    : {IMAGES_DIR}')
print(f'DSPy version  : {dspy.__version__}')


## Cell 2 — Configure DSPy (Pixtral) + Image Generation Clients

In [ ]:
import openai
from google import genai as google_genai
from google.genai import types as google_types
from dotenv import load_dotenv

load_dotenv(PROJECT_ROOT / '.env')

MISTRAL_API_KEY = os.environ['MISTRAL_API_KEY']
OPENAI_API_KEY  = os.environ['OPENAI_API_KEY']
GOOGLE_API_KEY  = os.environ['GOOGLE_API_KEY']

pixtral_lm = dspy.LM(
    'openai/pixtral-12b-2409',
    api_key=MISTRAL_API_KEY,
    api_base='https://api.mistral.ai/v1',
    temperature=0.2,
    max_tokens=500,
)
dspy.configure(lm=pixtral_lm)

dalle_client  = openai.OpenAI(api_key=OPENAI_API_KEY)
google_client = google_genai.Client(api_key=GOOGLE_API_KEY)

print(f'DSPy LM       : pixtral-12b-2409 via Mistral endpoint')
print(f'DALL-E client : key ...{OPENAI_API_KEY[-4:]}')
print(f'Gemini client : key ...{GOOGLE_API_KEY[-4:]}')


## Cell 3 — Models + Signature + Agent

- `ImageAlignmentResult` — typed output (`question_id`, `verdict`, `reasoning`)
- `ImageJudgeSignature` — `image_content: str` + `image: dspy.Image` → `output: ImageAlignmentResult`
- `ImageJudgeAgent` — wraps `dspy.ChainOfThought(ImageJudgeSignature)`

In [ ]:
from pydantic import BaseModel
from typing import Literal


class ImageAlignmentResult(BaseModel):
    question_id: str
    verdict:     Literal['aligned', 'not_aligned']
    reasoning:   str


class ImageJudgeSignature(dspy.Signature):
    """Judge whether the generated image correctly represents the described notice or sign.

    An image is 'aligned' when it visually depicts the notice/sign described in image_content
    and would logically support the correct answer to the question.

    An image is 'not_aligned' when it misrepresents, ignores, or contradicts the description,
    or contains content that would mislead rather than support the correct answer.

    Return verdict as exactly 'aligned' or 'not_aligned'.
    """
    image_content: str        = dspy.InputField(desc='Text description of the real-world notice or sign')
    image:         dspy.Image = dspy.InputField(desc='Generated image to evaluate against the description')
    output:        ImageAlignmentResult = dspy.OutputField(desc='Verdict (aligned/not_aligned) and brief reasoning')


class ImageJudgeAgent(dspy.Module):
    """Single-image alignment judge. Attribute `judge` serialises as `judge.predict` in the artifact."""

    def __init__(self):
        super().__init__()
        self.judge = dspy.ChainOfThought(ImageJudgeSignature)

    def forward(self, image_content: str, image, question_id: str = '') -> dspy.Prediction:
        return self.judge(image_content=image_content, image=image)


print('Models and agent defined.')
print('  Input  : image_content: str  +  image: dspy.Image')
print('  Output : output.verdict   -> aligned | not_aligned')
print('           output.reasoning -> brief explanation')


## Cell 4 — Define Metric

Score: `aligned == expected` -> 1.0, mismatch -> 0.0  
Returns `(float, str)` for GEPA — plain `float` for BootstrapFewShot.

In [ ]:
def alignment_metric(gold, pred, trace=None, pred_name=None, pred_trace=None):
    """
    aligned == expected  -> 1.0
    mismatch             -> 0.0
    Returns (float, str) for GEPA, plain float for BootstrapFewShot.
    """
    try:
        result    = pred.output
        predicted = result.verdict
    except Exception:
        msg = 'No valid output.'
        return (0.0, msg) if pred_name is not None else 0.0

    expected = gold.expected_verdict
    score    = 1.0 if predicted == expected else 0.0

    feedback = (
        f'Expected verdict={expected} (got "{predicted}"). Score: {score:.1f}. '
        + ('Correct.' if score == 1.0
           else 'Check whether the image visually matches the image_content description.')
    )
    return (score, feedback) if pred_name is not None else score


print('alignment_metric defined.')
print('  aligned == expected   -> 1.0')
print('  mismatch              -> 0.0')


## Cell 5 — Load Dataset

| Split | Questions | Examples (x2 models) |
|-------|-----------|----------------------|
| Train | Q1-Q20    | 80                   |
| Eval  | Q21-Q24   | 16                   |

In [ ]:
raw_data = json.loads(DATASET_PATH.read_text(encoding='utf-8'))
print(f'Loaded {len(raw_data)} questions from {DATASET_PATH.name}')
labels = [c['label'] for q in raw_data for c in q['candidates']]
print(f'Candidates   : {len(labels)} total  {dict(Counter(labels))}')

TRAIN_Q = raw_data[:20]   # Q1-Q20  (40 candidates x 2 models = 80 examples)
EVAL_Q  = raw_data[20:]   # Q21-Q24 (8 candidates x 2 models = 16 examples)

print(f'Train split  : {len(TRAIN_Q)} questions')
print(f'Eval split   : {len(EVAL_Q)} questions')
print()

for q in raw_data[:3]:
    ic = q['base_data']['image_content'][:70]
    print(f"  {q['question_id']}: {ic}...")


## Cell 6 — Generate Training Images

For each of the 48 candidates, generates one DALL-E PNG and one Gemini PNG.  
Saved to `data/image_mcq/images/judge_training/`.  
Safe to re-run — skips files that already exist.

In [ ]:
from PIL import Image as PILImage

def _generate_dalle(prompt: str, out_path: Path) -> bool:
    if out_path.exists(): return True
    try:
        resp = dalle_client.images.generate(
            model='dall-e-3', prompt=prompt,
            size='1024x1024', quality='standard', n=1, response_format='url',
        )
        urllib.request.urlretrieve(resp.data[0].url, out_path)
        return True
    except Exception as e:
        print(f'  [DALL-E FAILED] {out_path.name}: {e}')
        return False

def _generate_gemini(prompt: str, out_path: Path) -> bool:
    if out_path.exists(): return True
    try:
        resp = google_client.models.generate_images(
            model='imagen-3.0-generate-001', prompt=prompt,
            config=google_types.GenerateImagesConfig(number_of_images=1),
        )
        img_bytes = resp.generated_images[0].image.image_bytes
        PILImage.open(io.BytesIO(img_bytes)).save(out_path, format='PNG')
        return True
    except Exception as e:
        print(f'  [Gemini FAILED] {out_path.name}: {e}')
        return False

print('Generating training images (skips existing files)...')
generated, skipped, failed = 0, 0, 0

for q in raw_data:
    for c in q['candidates']:
        cid, prompt = c['candidate_id'], c['prompt']
        for prefix, fn in [('dalle', _generate_dalle), ('gemini', _generate_gemini)]:
            path = IMAGES_DIR / f'{prefix}_{cid}.png'
            if path.exists():
                skipped += 1
            else:
                ok = fn(prompt, path)
                generated += 1 if ok else 0
                failed    += 0 if ok else 1
                if ok: print(f'  Generated: {path.name}')

total_expected = len(raw_data) * 2 * 2
print(f'\nDone. Generated={generated}  Skipped={skipped}  Failed={failed}  Expected={total_expected}')


## Cell 7 — Build DSPy Examples

In [ ]:
def _png_to_dspy_image(path: Path) -> dspy.Image:
    b64 = base64.b64encode(path.read_bytes()).decode()
    return dspy.Image(url=f'data:image/png;base64,{b64}')

def _build_examples(questions: list) -> list:
    examples = []
    for q in questions:
        image_content = q['base_data']['image_content']
        for c in q['candidates']:
            cid, label = c['candidate_id'], c['label']
            for model_prefix in ('dalle', 'gemini'):
                img_path = IMAGES_DIR / f'{model_prefix}_{cid}.png'
                if not img_path.exists():
                    print(f'  [SKIP] Missing: {img_path.name}')
                    continue
                examples.append(
                    dspy.Example(
                        question_id      = f'{model_prefix}_{cid}',
                        image_content    = image_content,
                        image            = _png_to_dspy_image(img_path),
                        expected_verdict = label,
                        label            = label,
                    ).with_inputs('image_content', 'image')
                )
    return examples

trainset = _build_examples(TRAIN_Q)
evalset  = _build_examples(EVAL_Q)

print(f'Training examples : {len(trainset)}  (Q1-Q20)')
print(f'Eval examples     : {len(evalset)}   (Q21-Q24)')
print(f'Train labels : {dict(Counter(ex.label for ex in trainset))}')
print(f'Eval  labels : {dict(Counter(ex.label for ex in evalset))}')


## Cell 8 — Baseline Evaluation (Before Optimization)

In [ ]:
def evaluate_agent(agent, dataset, label=''):
    records = []
    for ex in dataset:
        try:
            pred   = agent(image_content=ex.image_content, image=ex.image, question_id=ex.question_id)
            score  = alignment_metric(ex, pred)
            result = pred.output
        except Exception as e:
            pred, score, result = None, 0.0, None
            print(f'  [ERROR] {ex.question_id}: {e}')
        records.append({'question_id': ex.question_id, 'gold': ex, 'result': result, 'score': score})

    avg = sum(r['score'] for r in records) / len(records) if records else 0.0
    print(f'[{label}]  N={len(records)}  avg_score={avg:.3f}')
    return records, avg

baseline_agent = ImageJudgeAgent()
print('Running baseline evaluation on training dataset...')
baseline_records, baseline_score = evaluate_agent(baseline_agent, trainset, label='Baseline')

print()
print(f'{"ID":<28} {"expected":<14} {"predicted":<14} {"score":>5}')
print('-' * 65)
for r in baseline_records:
    res = r['result']
    print(f"{r['question_id']:<28} {r['gold'].expected_verdict:<14} "
          f"{(res.verdict if res else 'ERR'):<14} {r['score']:>5.1f}")
print('-' * 65)
print(f'Baseline avg : {baseline_score:.3f}')


## Cell 9 — Run Optimizer

Tries **GEPA** first — rewrites the alignment prompt to improve verdict accuracy.  
Falls back to **BootstrapFewShot** if GEPA raises any error.  
Saves to `artifacts/image_judge_optimized.json`.

In [ ]:
from dotenv import load_dotenv
load_dotenv(dotenv_path=PROJECT_ROOT / '.env')


def run_gepa(trainset):
    from dspy.teleprompt import GEPA
    reflection_lm = dspy.LM(
        f"openai/{os.getenv('MISTRAL_MODEL', 'open-mistral-nemo')}",
        api_key=os.environ['MISTRAL_API_KEY'],
        api_base=os.getenv('MISTRAL_API_BASE', 'https://api.mistral.ai/v1'),
        temperature=0.9, max_tokens=4000,
    )
    log_dir = ARTIFACTS_DIR / 'gepa_image_judge_logs'
    log_dir.mkdir(exist_ok=True)
    optimizer = GEPA(
        metric=alignment_metric, auto='light', reflection_lm=reflection_lm,
        reflection_minibatch_size=3, log_dir=str(log_dir), track_stats=True, seed=42,
    )
    student = ImageJudgeAgent()
    split   = max(2, int(len(trainset) * 0.8))
    print(f'  GEPA: {split} train / {len(trainset) - split} val examples')
    optimized = optimizer.compile(student, trainset=trainset[:split], valset=trainset[split:])
    return optimized, 'GEPA'


def run_bootstrap(trainset):
    from dspy.teleprompt import BootstrapFewShot
    optimizer = BootstrapFewShot(metric=alignment_metric, max_bootstrapped_demos=3, max_labeled_demos=4)
    student   = ImageJudgeAgent()
    print(f'  BootstrapFewShot: {len(trainset)} examples')
    return optimizer.compile(student, trainset=trainset), 'BootstrapFewShot'


print('Attempting GEPA optimization...')
try:
    optimized_agent, optimizer_used = run_gepa(trainset)
    print('GEPA complete.')
except Exception as gepa_err:
    print(f'GEPA failed: {gepa_err}')
    print('Falling back to BootstrapFewShot...')
    optimized_agent, optimizer_used = run_bootstrap(trainset)
    print('BootstrapFewShot complete.')

optimized_agent.save(str(OPTIMIZED_PATH))
print(f'\nOptimizer used : {optimizer_used}')
print(f'Artifact saved : {OPTIMIZED_PATH}')


## Cell 10 — Load Optimized Agent & Post-Optimization Evaluation

In [ ]:
loaded_agent = ImageJudgeAgent()
loaded_agent.load(str(OPTIMIZED_PATH))
print(f'Loaded optimized agent from {OPTIMIZED_PATH}')

print('\nRunning post-optimization evaluation on training dataset...')
optimized_records, optimized_score = evaluate_agent(loaded_agent, trainset, label='Optimized')

print()
print(f'{"ID":<28} {"expected":<14} {"predicted":<14} {"score":>5}')
print('-' * 65)
for r in optimized_records:
    res = r['result']
    print(f"{r['question_id']:<28} {r['gold'].expected_verdict:<14} "
          f"{(res.verdict if res else 'ERR'):<14} {r['score']:>5.1f}")
print('-' * 65)
print(f'Baseline avg  : {baseline_score:.3f}')
print(f'Optimized avg : {optimized_score:.3f}')
print(f'Delta         : {optimized_score - baseline_score:+.3f}')


## Cell 11 — Write Promptfoo Provider

Standalone Python file imported by Promptfoo.  
Accepts JSON `{image_content, image_path}`, loads PNG as base64, runs optimized judge.

In [ ]:
# Cell 11 — Write Promptfoo Provider
import json as _json
_provider_code = _json.loads('"from __future__ import annotations\\nimport json, os, sys, base64\\nfrom pathlib import Path\\nfrom typing import Literal\\nfrom pydantic import BaseModel\\n\\nEVALS_DIR    = Path(__file__).resolve().parent\\nPROJECT_ROOT = EVALS_DIR.parent.parent\\nDATA_DIR     = PROJECT_ROOT / \'data\' / \'image_mcq\'\\nIMAGES_DIR   = DATA_DIR / \'images\' / \'judge_training\'\\n\\nif str(PROJECT_ROOT) not in sys.path:\\n    sys.path.insert(0, str(PROJECT_ROOT))\\n\\n_venv_sp = PROJECT_ROOT / \'.venv\' / \'Lib\' / \'site-packages\'\\nif not _venv_sp.exists():\\n    _venv_sp = PROJECT_ROOT / \'.venv\' / \'lib\' / \'site-packages\'\\nif _venv_sp.exists() and str(_venv_sp) not in sys.path:\\n    sys.path.insert(0, str(_venv_sp))\\n\\nfrom dotenv import load_dotenv\\nload_dotenv(PROJECT_ROOT / \'.env\')\\nimport dspy\\n\\n\\nclass ImageAlignmentResult(BaseModel):\\n    question_id: str\\n    verdict:     Literal[\'aligned\', \'not_aligned\']\\n    reasoning:   str\\n\\n\\nclass ImageJudgeSignature(dspy.Signature):\\n    # Judge: \'aligned\' = image matches description; \'not_aligned\' = does not match.\\n    image_content: str        = dspy.InputField(desc=\'Text description of the notice/sign\')\\n    image:         dspy.Image = dspy.InputField(desc=\'Generated image to evaluate\')\\n    output:        ImageAlignmentResult = dspy.OutputField(desc=\'Verdict and reasoning\')\\n\\n\\nclass ImageJudgeAgent(dspy.Module):\\n    def __init__(self):\\n        super().__init__()\\n        self.judge = dspy.ChainOfThought(ImageJudgeSignature)\\n\\n    def forward(self, image_content, image, question_id=\'\'):\\n        return self.judge(image_content=image_content, image=image)\\n\\n\\n_OPTIMIZED = PROJECT_ROOT / \'artifacts\' / \'image_judge_optimized.json\'\\n_agent = None\\n\\n\\ndef call_api(prompt: str, options, context):\\n    global _agent\\n    lm = dspy.LM(\\n        \'openai/pixtral-12b-2409\',\\n        api_key=os.environ[\'MISTRAL_API_KEY\'],\\n        api_base=\'https://api.mistral.ai/v1\',\\n        temperature=0.2, max_tokens=500,\\n    )\\n    dspy.configure(lm=lm)\\n    if _agent is None:\\n        _agent = ImageJudgeAgent()\\n        if _OPTIMIZED.exists():\\n            _agent.load(str(_OPTIMIZED))\\n    try:\\n        data          = json.loads(prompt)\\n        image_content = data[\'image_content\']\\n        img_path      = IMAGES_DIR / data[\'image_path\']\\n        b64           = base64.b64encode(img_path.read_bytes()).decode()\\n        img           = dspy.Image(url=f\'data:image/png;base64,{b64}\')\\n    except Exception as e:\\n        return {\'error\': f\'Invalid input: {e}\'}\\n    try:\\n        pred   = _agent(image_content=image_content, image=img)\\n        result = pred.output.model_dump()\\n        return {\'output\': json.dumps(result)}\\n    except Exception as e:\\n        return {\'error\': str(e)}"')
provider_path = EVALS_DIR / 'image_judge_eval_provider.py'
provider_path.write_text(_provider_code, encoding='utf-8')
print(f'Written: {provider_path}')

## Cell 12 — Build Promptfoo Test Cases

Source: eval split Q21-Q24 (both DALL-E and Gemini images, both labels).  
Each test asserts `verdict` matches ground truth label.

In [ ]:
def build_tests(questions: list) -> list:
    tests = []
    for q in questions:
        image_content = q['base_data']['image_content']
        for c in q['candidates']:
            cid, label = c['candidate_id'], c['label']
            for model_prefix in ('dalle', 'gemini'):
                img_filename = f'{model_prefix}_{cid}.png'
                if not (IMAGES_DIR / img_filename).exists():
                    continue
                input_obj = {'image_content': image_content, 'image_path': img_filename}
                tests.append({
                    'description': f'{model_prefix}_{cid} | expect={label}',
                    'vars': {'image_input': json.dumps(input_obj)},
                    'assert': [{'type': 'javascript',
                                'value': f'const p = JSON.parse(output); return p.verdict === "{label}";'}],
                })
    return tests

all_tests = build_tests(EVAL_Q)
print(f'Eval test cases : {len(all_tests)}  (Q21-Q24, both models)')
label_dist = Counter(t['description'].split('expect=')[1] for t in all_tests)
print(f'Label dist      : {dict(label_dist)}')
print()
for t in all_tests:
    print(f'  {t["description"]}')


## Cell 13 — Write Promptfoo Config YAML

In [ ]:
import yaml

config = {
    'description': 'ImageJudge eval — verdict (aligned / not_aligned) per image',
    'providers': [{'id': 'file://image_judge_eval_provider.py'}],
    'prompts': ['{{image_input}}'],
    'tests': all_tests,
}

config_path = EVALS_DIR / 'image_judge_eval_config.yaml'
with config_path.open('w', encoding='utf-8') as f:
    yaml.dump(config, f, allow_unicode=True, default_flow_style=False, sort_keys=False)

print(f'Written: {config_path}')


## Cell 14 — Run Promptfoo Eval as Subprocess

In [ ]:
cmd = [
    'npx', 'promptfoo@latest', 'eval',
    '-c', str(config_path),
    '-o', str(EVAL_OUTPUT),
    '--output-format', 'json',
    '--no-cache',
    '--max-concurrency', '1',
]

print('Running Promptfoo eval...')
result = subprocess.run(cmd, capture_output=True, text=True, shell=True, cwd=str(EVALS_DIR), timeout=600)

stdout = result.stdout
print(stdout[-3000:] if len(stdout) > 3000 else stdout)
if result.returncode != 0:
    stderr = result.stderr
    print(stderr[-2000:] if len(stderr) > 2000 else stderr)
    print(f'\nPromptfoo exited with code {result.returncode}')
else:
    print('\nPromptfoo completed successfully (exit code 0).')


## Cell 15 — Parse & Display Promptfoo Results

In [ ]:
if not EVAL_OUTPUT.exists():
    print(f'Output file not found: {EVAL_OUTPUT}')
else:
    raw_text = EVAL_OUTPUT.read_text(encoding='utf-8').strip()
    if not raw_text:
        print('Output file is empty.')
    else:
        try:
            eval_data = json.loads(raw_text)
        except json.JSONDecodeError as e:
            print(f'Parse error: {e}'); print(raw_text[:500])
        else:
            raw          = eval_data.get('results', {})
            results_list = raw.get('results', raw) if isinstance(raw, dict) else raw
            stats        = raw.get('stats', {}) if isinstance(raw, dict) else {}

            passed = sum(1 for r in results_list if r.get('success', False))
            total  = len(results_list)
            print(f'Promptfoo Eval Results')
            print(f'  Passed : {passed}/{total}')
            print(f'  Failed : {total - passed}/{total}')
            if stats: print(f'  Stats  : {stats}')
            print()

            print(f'{"Test ID":<30} {"Result":<7} {"predicted":<14} {"expected"}')
            print('-' * 70)
            for r in results_list:
                desc     = r.get('description', '')[:29]
                success  = 'PASS' if r.get('success') else 'FAIL'
                expected = desc.split('expect=')[-1] if 'expect=' in desc else 'N/A'
                try:
                    p       = json.loads(r.get('response', {}).get('output', '{}'))
                    verdict = p.get('verdict', 'N/A')
                except Exception:
                    verdict = 'PARSE_ERR'
                print(f'{desc:<30} {success:<7} {verdict:<14} {expected}')
            print()
            print(f'Full eval output saved to: {EVAL_OUTPUT}')
